# Custom High-Scale Wake Word Training Pipeline

**Target phrase**: Hello DJ  
**Model name**: Hello_DJ (auto-derived from target_phrase spaces).  
**Scale**: Matched to hey_jarvis production config - 200k positive samples, 50k training steps.

In [ ]:
# Step 1: Download MIT RIRs and setup background audio datasets
import os, shutil
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm import tqdm

# --- Download MIT RIRs from HuggingFace Hub ---
repo_id = "davidscripka/MIT_environmental_impulse_responses"
output_dir = "/home/jovyan/mit_rirs/16khz"
os.makedirs(output_dir, exist_ok=True)

print("Downloading MIT environmental impulse responses...")
rir_files = [f for f in list_repo_files(repo_id, repo_type='dataset')
             if f.startswith('16khz/') and f.endswith('.wav')]

for fname in tqdm(rir_files, desc="MIT RIRs"):
    hf_hub_download(
        repo_id=repo_id,
        filename=fname,
        repo_type='dataset',
        local_dir="/home/jovyan/mit_rirs"
    )

n_rirs = len([f for f in os.listdir(output_dir) if f.endswith('.wav')])
print(f"mit_rirs: {n_rirs} wav files in 16khz/")

# --- Copy tutorial positive clips as background audio ---
os.makedirs('/tmp/audioset_16k', exist_ok=True)
src_pos = '/home/jovyan/open-wakeword-repo/notebooks/training_tutorial_data/positive'
if os.path.isdir(src_pos):
    for w in [f for f in os.listdir(src_pos) if f.endswith('.wav')]:
        shutil.copy(os.path.join(src_pos, w), '/tmp/audioset_16k/')
print(f'audioset_16k: {len(os.listdir("/tmp/audioset_16k"))} wav files')

# --- Also copy piper-sample-generator impulses as additional RIRs ---
impulse_src = '/home/jovyan/piper-sample-generator/impulses'
if os.path.isdir(impulse_src):
    for w in os.listdir(impulse_src):
        if w.endswith('.wav'):
            shutil.copy(os.path.join(impulse_src, w), output_dir)
print(f'mit_rirs after impulses copy: {len([f for f in os.listdir(output_dir) if f.endswith(".wav")])} wav files')

In [ ]:
# Step 2: Download pre-computed openWakeWord features
import os
from huggingface_hub import hf_hub_download

HF_TOKEN = None

print("Downloading pre-computed openWakeWord features...")

# Validation set features (needed for false positive rate estimation during training)
hf_hub_download(
    repo_id="davidscripka/openwakeword_features",
    filename="validation_set_features.npy",
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir="/home/jovyan"
)

# Training set features (~2000 hours from ACAV100M dataset)
hf_hub_download(
    repo_id="davidscripka/openwakeword_features",
    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    repo_type="dataset",
    token=HF_TOKEN,
    local_dir="/home/jovyan"
)

print("Pre-computed features downloaded:")
val_path = "/home/jovyan/validation_set_features.npy"
feat_path = "/home/jovyan/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
print(f"  validation_set_features.npy: {os.path.getsize(val_path) / 1e6:.1f} MB")
print(f"  openwakeword_features_ACAV100M_2000_hrs_16bit.npy: {os.path.getsize(feat_path) / 1e6:.1f} MB")

## Step 3: Setup openwakeword and dependencies

In [ ]:
# Step 3a: Install openwakeword, patch piper, and download model config
import sys, os

# Ensure openwakeword is importable from the local clone (has train.py)
sys.path.insert(0, "/home/jovyan/openwakeword")

# Verify openwakeword is installed
try:
    import openwakeword
    print(f"openwakeword {openwakeword.__version__} already installed at {openwakeword.__file__}")
except ImportError:
    get_ipython().system("pip install -e /home/jovyan/openwakeword 2>&1 | tail -5")

# Install piper dependencies
get_ipython().system("pip install --user --break-system-packages webrtcvad espeak-phonemizer 2>&1 | tail -3")
get_ipython().system("pip install 'numpy<2' piper-tts 2>&1 | tail -3")
print("Dependencies OK")

# --- Patch generate_samples.py for PyTorch 2.6+ weights_only ---
gen_path = '/home/jovyan/piper-sample-generator/generate_samples.py'
with open(gen_path, 'r') as f:
    content = f.read()
content = content.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)')
with open(gen_path, 'w') as f:
    f.write(content)
print("Patched generate_samples.py for weights_only=False")

# --- Download model JSON config if missing ---
import requests
models_dir = '/home/jovyan/piper-sample-generator/models'
json_path = os.path.join(models_dir, 'en-us-libritts-high.pt.json')
if not os.path.exists(json_path) or os.path.getsize(json_path) == 0:
    print("Downloading model JSON config...")
    url = 'https://raw.githubusercontent.com/rhasspy/piper-sample-generator/v2.0.0/models/en-us-libritts-high.pt.json'
    r = requests.get(url)
    with open(json_path, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded {len(r.content)} bytes")
else:
    print(f"Model JSON config exists ({os.path.getsize(json_path)} bytes)")

# --- Fix data.py to use exist_ok ---
import_path = '/home/jovyan/openwakeword/openwakeword/data.py'
with open(import_path, 'r') as f:
    content = f.read()
content = content.replace(
    'os.mkdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"))',
    'os.makedirs(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"), exist_ok=True)'
)
with open(import_path, 'w') as f:
    f.write(content)
print("Patched data.py with exist_ok=True")

# --- Copy generate_samples.py and piper_train into openwakeword ---
get_ipython().system("cp /home/jovyan/piper-sample-generator/generate_samples.py /home/jovyan/openwakeword/openwakeword/generate_samples.py")
get_ipython().system("cp -r /home/jovyan/piper-sample-generator/piper_train /home/jovyan/openwakeword/openwakeword/piper_train")
print("Copied generate_samples.py and piper_train module")

In [ ]:
# Step 3b: Read, update, and save training config
import yaml

with open('/home/jovyan/my_custom_model.yml') as f:
    config = yaml.safe_load(f.read())

print("Current config:")
print(yaml.dump(config))

# Update config with correct paths and reduced batch size to avoid GPU OOM
config.update({
    'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
    'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
    'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
    'background_paths': ['/tmp/audioset_16k'],
    'background_paths_duplication_rate': [1],
    'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
    'output_dir': '/home/jovyan/Hello_DJ',
    'model_name': 'Hello_DJ',
    'target_phrase': ['Hello DJ'],
    'n_samples': 200000,
    'steps': 50000,
    'layer_size': 64,
    'augmentation_rounds': 2,
    'tts_batch_size': 32,
    'target_false_positives_per_hour': 0.2,
    'max_negative_weight': 1500,
    'n_samples_val': 5000,
    'custom_negative_phrases': [],
    'augmentation_batch_size': 16,
    'feature_data_files': {
        'ACAV100M_sample': '/home/jovyan/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    },
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50
    },
})

with open('/home/jovyan/my_custom_model.yml', 'w') as f:
    yaml.dump(config, f)

print("\nUpdated config saved:")
print(yaml.dump(config))

## Step 4: Generate synthetic training clips

Runs Piper TTS to generate 200k positive samples of "Hello DJ" and adversarial negative samples.

In [ ]:
# Step 4a: Generate positive and negative synthetic clips
import sys

sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-12.9/compat:$LD_LIBRARY_PATH \
    PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH \
    python3 /home/jovyan/openwakeword/openwakeword/train.py \
    --training_config /home/jovyan/my_custom_model.yml \
    --generate_clips

In [ ]:
# Step 4b: Augment clips with RIRs and background noise
import sys

sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-12.9/compat:$LD_LIBRARY_PATH \
    PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH \
    python3 /home/jovyan/openwakeword/openwakeword/train.py \
    --training_config /home/jovyan/my_custom_model.yml \
    --augment_clips

## Step 5: Train the model

Requires augmented features from Step 4b.

In [ ]:
# Step 5a: Train the model (50k steps, 64-unit layers)
import sys

sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-12.9/compat:$LD_LIBRARY_PATH \
    PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH \
    python3 /home/jovyan/openwakeword/openwakeword/train.py \
    --training_config /home/jovyan/my_custom_model.yml \
    --train_model

In [ ]:
# Step 5b: Verify outputs
import os, glob

print("=== Output check ===")
model_dir = '/home/jovyan/Hello_DJ/Hello_DJ'
if os.path.isdir(model_dir):
    for sub in ['positive_train', 'positive_test', 'negative_train', 'negative_test']:
        path = os.path.join(model_dir, sub)
        n = len(glob.glob(os.path.join(path, '*.wav'))) if os.path.isdir(path) else 0
        print(f"  {sub}: {n} wav files")

onnx_path = os.path.join('/home/jovyan/Hello_DJ', 'Hello_DJ.onnx')
tflite_path = os.path.join('/home/jovyan/Hello_DJ', 'Hello_DJ.tflite')
if os.path.exists(onnx_path):
    print(f"  ONNX model: {onnx_path} ({os.path.getsize(onnx_path) / 1e6:.1f} MB)")
if os.path.exists(tflite_path):
    print(f"  TFLite model: {tflite_path} ({os.path.getsize(tflite_path) / 1e6:.1f} MB)")
print("Done")